# 📘 Module 7.2 — End-to-End Driving Models

**Goal:** Understand end-to-end autonomous driving — from pixels to control.

## Modular vs. End-to-End

```
MODULAR PIPELINE (Traditional):
Sensors → Perception → Prediction → Planning → Control
         (detect)     (forecast)   (route)    (steer)
         
Each module designed separately. Errors propagate through pipeline.

END-TO-END (Modern):
Sensors ──────────────────────────────────────→ Control
         Single neural network learns everything
```

---

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. Behavioral Cloning — The Simplest E2E Approach

Learn to drive by **imitating human drivers**:
- Collect data: (camera image, steering angle) pairs
- Train CNN to predict steering from images
- This was NVIDIA's approach in their 2016 paper!

In [2]:
class PilotNet(nn.Module):
    """
    NVIDIA PilotNet (2016) — one of the first E2E driving models.
    Predicts steering angle from a front-camera image.
    Paper: "End to End Learning for Self-Driving Cars"
    """
    
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            # Input: 3 × 66 × 200 (NVIDIA's original input size)
            nn.Conv2d(3, 24, 5, stride=2),   # → 24 × 31 × 98
            nn.ReLU(),
            nn.Conv2d(24, 36, 5, stride=2),  # → 36 × 14 × 47
            nn.ReLU(),
            nn.Conv2d(36, 48, 5, stride=2),  # → 48 × 5 × 22
            nn.ReLU(),
            nn.Conv2d(48, 64, 3),            # → 64 × 3 × 20
            nn.ReLU(),
            nn.Conv2d(64, 64, 3),            # → 64 × 1 × 18
            nn.ReLU(),
        )
        
        self.controller = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 1 * 18, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 10),
            nn.ReLU(),
            nn.Linear(10, 1),  # Single output: steering angle
        )
    
    def forward(self, x):
        x = self.features(x)
        steering = self.controller(x)
        return steering

model = PilotNet()
img = torch.randn(1, 3, 66, 200)
steering = model(img)

print(f"PilotNet: Image {img.shape} → Steering angle {steering.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nThis was groundbreaking in 2016 — a CNN directly controlling a car!")

PilotNet: Image torch.Size([1, 3, 66, 200]) → Steering angle torch.Size([1, 1])
Parameters: 252,219

This was groundbreaking in 2016 — a CNN directly controlling a car!


## 2. Modern E2E: Multi-Camera + Transformer

In [3]:
class ModernE2E(nn.Module):
    """
    Modern end-to-end driving model (simplified).
    Inspired by UniAD, VAD, and Tesla's architecture.
    """
    
    def __init__(self, num_cameras=6, d_model=256, num_waypoints=10):
        super().__init__()
        
        # Per-camera feature extractor
        self.camera_backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=4, padding=3),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, d_model),
        )
        
        # Cross-camera fusion (transformer)
        self.camera_fusion = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=8, dim_feedforward=1024, batch_first=True),
            num_layers=3
        )
        
        # Ego state encoder
        self.state_proj = nn.Linear(8, d_model)  # speed, accel, heading, etc.
        
        # Prediction heads
        self.waypoint_head = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.ReLU(),
            nn.Linear(512, num_waypoints * 2),  # (x, y) for each waypoint
        )
        
        self.control_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 3),  # steering, throttle, brake
        )
        
        self.num_cameras = num_cameras
        self.num_waypoints = num_waypoints
    
    def forward(self, cameras, ego_state):
        """
        Args:
            cameras: (B, num_cameras, 3, H, W)
            ego_state: (B, 8)
        """
        B = cameras.shape[0]
        
        # Process each camera
        cam_features = []
        for i in range(self.num_cameras):
            feat = self.camera_backbone(cameras[:, i])  # (B, d_model)
            cam_features.append(feat)
        
        cam_tokens = torch.stack(cam_features, dim=1)  # (B, num_cameras, d_model)
        
        # Add ego state
        state_token = self.state_proj(ego_state).unsqueeze(1)  # (B, 1, d_model)
        tokens = torch.cat([cam_tokens, state_token], dim=1)
        
        # Fuse
        fused = self.camera_fusion(tokens)
        pooled = fused.mean(dim=1)  # (B, d_model)
        
        # Predict
        waypoints = self.waypoint_head(pooled).view(B, self.num_waypoints, 2)
        controls = self.control_head(pooled)
        controls = torch.cat([
            torch.tanh(controls[:, :1]),     # Steering: [-1, 1]
            torch.sigmoid(controls[:, 1:2]), # Throttle: [0, 1]
            torch.sigmoid(controls[:, 2:]),  # Brake: [0, 1]
        ], dim=1)
        
        return waypoints, controls

# Test with 6 cameras (front, rear, left, right, front-left, front-right)
model = ModernE2E(num_cameras=6)
cameras = torch.randn(2, 6, 3, 224, 224)  # 6 cameras
ego_state = torch.randn(2, 8)  # speed, accel, yaw, etc.

waypoints, controls = model(cameras, ego_state)

print(f"6 Camera inputs: {cameras.shape}")
print(f"Ego state: {ego_state.shape}")
print(f"Waypoints (trajectory): {waypoints.shape}")
print(f"Controls [steer, throttle, brake]: {controls.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

6 Camera inputs: torch.Size([2, 6, 3, 224, 224])
Ego state: torch.Size([2, 8])
Waypoints (trajectory): torch.Size([2, 10, 2])
Controls [steer, throttle, brake]: torch.Size([2, 3])
Parameters: 3,154,583


In [4]:
# --- Compare Approaches ---
print("Evolution of End-to-End Driving:")
print("=" * 60)
timeline = [
    ("2016", "NVIDIA PilotNet", "1 camera → steering angle"),
    ("2019", "ChauffeurNet (Waymo)", "Rendered top-down → trajectory"),
    ("2021", "Transfuser", "Camera + LiDAR → waypoints"),
    ("2022", "TCP/InterFuser", "Multi-view → controls + safety"),
    ("2023", "UniAD", "Unified perception + prediction + planning"),
    ("2024", "VAD/SparseDrive", "Vectorized + sparse representations"),
    ("2024", "EMMA (Waymo)", "Multimodal LLM for driving"),
    ("2025", "VLA-based", "Language-conditioned E2E driving"),
]

for year, model, desc in timeline:
    print(f"  {year}: {model:<20} — {desc}")

Evolution of End-to-End Driving:
  2016: NVIDIA PilotNet      — 1 camera → steering angle
  2019: ChauffeurNet (Waymo) — Rendered top-down → trajectory
  2021: Transfuser           — Camera + LiDAR → waypoints
  2022: TCP/InterFuser       — Multi-view → controls + safety
  2023: UniAD                — Unified perception + prediction + planning
  2024: VAD/SparseDrive      — Vectorized + sparse representations
  2024: EMMA (Waymo)         — Multimodal LLM for driving
  2025: VLA-based            — Language-conditioned E2E driving


---
## ✅ Key Takeaways

1. **End-to-end driving** maps sensor input directly to vehicle control
2. **Behavioral cloning** learns by imitating human drivers
3. Modern E2E uses **multi-camera** input with **transformer fusion**
4. Outputs include both **trajectory waypoints** and **vehicle controls**
5. The field is rapidly evolving — from simple CNNs to multimodal VLAs

---
## 📖 Next Steps
➡️ **Next module:** [08_ADAS_Applications/01_lane_detection.ipynb](../08_ADAS_Applications/01_lane_detection.ipynb) — Practical ADAS applications